<img src="images/scilife_logo.png" width="400">
<img src="images/essence_logo.png" width="300">

# SciLifeLab Workshop - Hands-on Section: LangGraph "Hello World"



Welcome to this hands‑on lab session on building AI Agents with **LangGraph**! 

LangGraph is a low‑level orchestration framework for constructing stateful AI workflows using graphs. LangGraph provides several key benefits for agentic AI applications, including durable execution, support for human‑in‑the‑loop workflows, comprehensive memory (both short‑ and long‑term), built‑in debugging, and production‑ready deployment features.

## Learning Objectives
By the end of this workshop, you will:
- Understand core concepts of LangGraph (tools, nodes, edges, state, and memory).
- Create and integrate your own tools for an AI agent.
- Build a ReAct‑style agent using an LLM and custom tools.
- Implement agent memory to maintain conversational context.
- Compare custom agents with prebuilt LangGraph agents.
- Explore extension tasks such as custom graph, structured output and prompts template

## Workshop Outline 
- **Part 1**: Setup & imports
- **Part 2**: Understanding and creating tools
- **Part 3**: Defining the state
- **Part 4**: Building the agent graph
- **Part 5**: Testing the agent
- **Part 6**: Adding short‑term memory
- **Part 7**: Exploring prebuilt agents
- **Part 8 (optional)**: Extension exercises

<img src="images/outline.png" width="800">

## Instructions for Participants

**Throughout this lab, look for `TODO` comments and `...` placeholders in the code cells. These indicate where you need to add your implementation.** 

Your task is to:
1. Replace `...` placeholders with appropriate code
2. Follow the instructions in `TODO` comments 
3. Refer to the exercise descriptions and API references provided

**Tip:** Each exercise builds on the previous one, so complete them in order!

---

## Part 1 – Setup

In this first step, we'll import the necessary dependencies and load any environment variables.

### Exercise 1.1 – Import dependencies

In [ ]:
# Import any packages you need here
import json
from dotenv import load_dotenv
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors
import requests
import re
import time
import pubchempy as pcp
from chembl_webresource_client.new_client import new_client
import pandas as pd
from urllib.parse import quote
from langchain.tools import tool

load_dotenv(override=True)

### Exercise 1.2 – Configure the LLM

In this workshop we use the **SciLifeLab pilot LLM service** by default (model `qwen3`).
The helper `get_llm()` below helps you call the LLM that will be used for agent later. It will tries the pilot service first; if it cannot be reached during the session (or no `PILOT_API_KEY` is set), it automatically falls back to OpenAI. 

From here on we
always create models with `get_llm()` instead of calling `ChatOpenAI(...)` directly, so
the same default + fallback logic applies everywhere.

In [ ]:
import os
from langchain_openai import ChatOpenAI

# LLM configuration: Default by SciLifeLab pilot LLM service. Fallback: OpenAI.
SCILIFELAB_PILOT_URL = "https://open-llm.scilifelab.se/api"
PILOT_MODEL = "qwen3"        # or "gemma3-27b"
OPENAI_MODEL = "gpt-5.4"     # or other OpenAI model


def get_llm(temperature: float = 0, **kwargs) -> ChatOpenAI:
    """Return a chat model, preferring the SciLifeLab pilot service.

    Order of preference:
      1. SciLifeLab pilot LLM (`PILOT_MODEL`) — used by default.
      2. OpenAI (`OPENAI_MODEL`) — fallback if the pilot service can't be
         reached during the session, or if no `PILOT_API_KEY` is set.
    """
    pilot_key = os.getenv("PILOT_API_KEY")
    openai_key = os.getenv("OPENAI_API_KEY")

    # 1) TODO: Try the pilot service first
    if pilot_key:
        pilot_llm = ChatOpenAI(
            model=PILOT_MODEL,
            base_url=...,
            api_key=pilot_key,
            temperature=temperature,
            **kwargs,
        )
        try:
            pilot_llm.invoke("ping")  # lightweight connectivity check
            print(f"✓ Using SciLifeLab pilot LLM ({PILOT_MODEL})")
            return pilot_llm
        except Exception as e:
            print(f"⚠ Pilot LLM unavailable ({type(e).__name__}: {e}). Falling back to OpenAI.")
    else:
        print("⚠ PILOT_API_KEY not set — falling back to OpenAI.")

    # 2) Fall back to OpenAI
    if not openai_key:
        raise RuntimeError(
            "No usable LLM: the pilot service failed and OPENAI_API_KEY is not set. "
            "Set PILOT_API_KEY and/or OPENAI_API_KEY in your .env file."
        )
    print(f"✓ Using OpenAI fallback ({OPENAI_MODEL})")
    return ChatOpenAI(model=..., temperature=temperature, **kwargs) # Fallback to OpenAI model

---

## Part 2 – Understanding and Creating Tools

In LangGraph, **tools** are Python functions that extend your agent's capabilities beyond text generation. They can call external APIs or perform computations, and are annotated with the `@tool` decorator from LangChain. 

When designing tools for agents, there are two important considerations: 
- **Input - Output**: It is similar to when designing a Python function, but the input is now generated by LLM (rather than human typing). Output should be parsed to feed meaningful context to the agent.

- **Description**: The `@tool` decorator requires docstrings from a Python function and will use it as the tool's description. These descriptions will feed the LLM system prompts, making it aware of these existing tools and their schema.

<img src="images/exercise_2_intro.png" width="800">

### About the task

In this exercise, we will develop three simple tools, which could be useful for the drug discovery.

**1. SMILES Resolver:** a tool to get SMILES string for compound from any identifers

**2. Get Properties:** a tool to get basic physiochemical properties of the molecule

**3. Lit Search:** a tool to perform semantic search among all publications available on PubMed




In [ ]:
# Initialize connection to external API
CACTUS_BASE = "https://cactus.nci.nih.gov/chemical/structure"
molecule = new_client.molecule

# Helper functions
def _is_chembl_id(value: str) -> bool:
    return bool(re.fullmatch(r"CHEMBL\d+", value.upper()))

def _looks_like_inchikey(value: str) -> bool:
    return bool(re.fullmatch(r"[A-Z]{14}-[A-Z]{10}-[A-Z]", value))


# Main tool
@tool
def resolve_smiles(
    identifier: str,
    pause_s: float = 0.0,
    timeout_s: float = 15.0,
) -> dict | None:
    """
    Resolve any identifier types to canonical SMILES.
    
    Args:
        identifier: any identifiers of compounds, i.e. name, CID, CHEMBL, CAS, etc.
    
    Returns: 
        dict: original identifiers and corresponding SMILES string
    """
    ident = identifier.strip()
    if not ident:
        return None
    
    ident_lower = ident.lower()
    ident_upper = ident.upper()
    
    # Try ChEMBL
    if _is_chembl_id(ident_upper):
        try:
            mol = molecule.get(ident_upper)
            smiles = mol.get("molecule_structures", {}).get("canonical_smiles")
            if smiles:
                if pause_s:
                    time.sleep(pause_s)
                return {'identifier': identifier, 'SMILES': smiles}
        except Exception:
            pass
    
    
    # Try CACTUS
    try:
        url = f"{CACTUS_BASE}/{quote(ident)}/smiles"
        response = requests.get(url, timeout=timeout_s, headers={"User-Agent": "smiles-resolver/1.0"})
        if response.ok:
            text = response.text.strip()
            # Explicitly check for non-empty and valid content
            if (text  and len(text) > 0 ):
                if pause_s:
                    time.sleep(pause_s)
                return {'identifier': identifier, 'SMILES': text}
            # If empty or invalid, fall through to PubChem
    except requests.RequestException:
        pass
    
    # Try PubChem
    try:
        if ident.isdigit():
            compound = pcp.Compound.from_cid(int(ident))
            smiles = getattr(compound, "canonical_smiles", None)
            if smiles:
                if pause_s:
                    time.sleep(pause_s)
                return {'identifier': identifier, 'SMILES': smiles}
        
        if _looks_like_inchikey(ident_upper):
            compounds = pcp.get_compounds(ident_upper, namespace="inchikey")
        else:
            compounds = pcp.get_compounds(ident_lower, namespace="name")
        
        if compounds:
            smiles = getattr(compounds[0], "canonical_smiles", None)
            if smiles:
                if pause_s:
                    time.sleep(pause_s)
                return {'identifier': identifier, 'SMILES': smiles}
    except Exception:
        pass
    
    return None

<img src="images/tool2.png" width="800">


For those who are not familiar with RDKit: 
- RDKit documentation: https://www.rdkit.org/docs/source/rdkit.Chem.Descriptors.html

In [ ]:
@tool 
def get_properties(smiles: str):
    # TODO: Write description for this function
    """
    What does it do?

    What are the args?

    What are the output (return)?
    """
    mol = Chem.MolFromSmiles(smiles)
    
    if mol is None:
        return {"error": "Invalid SMILES string"}
    
    # TODO: Get all desired properties
    # You can get more properties if you want
    properties = {
        "Molecular_Weight": Descriptors..., # get Molecular Weight
        "LogP": Descriptors..., # get Molecular logP
        "HBD": Descriptors...,  # get number of hydrogen bond donors
        "HBA": Descriptors...,  # get number of hydrogen bond acceptors
        "TPSA": Descriptors...,  # get TPSA
        "Rotatable_Bonds": Descriptors..., # get number of rotatable bonds
        "Aromatic_Rings": Descriptors..., # get number of aromatic rings
        "Heavy_Atoms": Descriptors..., # get number of heavy atoms
        "Formal_Charge": Chem.GetFormalCharge(mol),
    }
    
    return {
        'SMILES': smiles,
        'Properties': properties
    }


In [ ]:
from langchain.tools import tool
from utils.litsense.litsense import LitSense_API

@tool
def lit_search(query: str, limit: int = 20) -> str:
    """Retrieve information from PubMed using a semantic search via the LitSense API.\n\n
    
    Args:
        query: The research question or topic to search for in PubMed literature.
        limit: Maximum number of results to return (default is 5).
    
    Returns:
        str: all the paragraphs + corresponding PMID number.
    """

    try:
        engine = LitSense_API()
        results = engine.retrieve(query, limit=limit, rerank=False)
        if not results:
            return f"No relevant literature found for '{query}'. Please try a different or broader search query."
        if 'error' in results:
            return("  LitSense search failed:", results['error'])
        result_str = ""
        for i, result in enumerate(results):
            result_str += (
                f"\n--- Passage #{i+1} ---\n"
                f"PMID: {result.pmid}\n"
                f"Content: {result.text}\n"
            )
        return result_str
    except Exception as e:
        return f"Error retrieving literature for '{query}': {str(e)}. Please try a different search query."

### Exercise 2.4 – Create a Tools List

Collect all of your tool functions into a single list called `TOOLS`. This list will be passed to the LLM so that it is aware of what tools are available.

In [ ]:
# TODO: Put all your tools function into TOOLS list
TOOLS = [...]

---

## Part 3 – Understanding LangGraph State

For agents, the state typically contains a list of messages which grows over the conversation. 

When creating a state schema, you use `TypedDict` to describe the keys and `Annotated` with updating functions to specify how values should be updated. 

The `add_messages` function appends new messages rather than overwriting them.


### Exercise 3.1 – Define Chat State

Define a `ChatState` class (subclassing `TypedDict`) with a single key `messages`. Use the `add_messages` updating function so that new messages are appended to the state.

<img src="images/ChatState.png" width="800">

In [ ]:
from typing_extensions import TypedDict
from typing import Annotated, List, Dict
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    """The state schema for our LangGraph. It contains only a list of messages.

    Messages are appended via the `add_messages` to preserve the full conversation history."""
    messages: Annotated[List[Dict], add_messages]

---

## Part 4 – Building the Agent Graph

This is the graph we are going to create: 

<img src="images/react_agent.png" width="300">

### Exercise 4.1 – Initialize Components

1. Initialise a `StateGraph` instance using your `ChatState`.
2. Initialise a chat model with the `get_llm()` helper from Part 1 (it defaults to the SciLifeLab pilot LLM, with OpenAI as a fallback).
3. Bind your tools to the LLM using `bind_tools()`.
4. Create a `ToolNode` from your tools list.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

# Create graph builder using ChatState
graph_builder = StateGraph(ChatState)

# TODO: Initialise the chat model using the get_llm() helper from Part 1
llm = ...

# TODO: Bind the tools to the LLM so that it knows their schemas and how to construct tool calls in JSON
llm_with_tools = llm.bind_tools(..., parallel_tool_calls=False)

# TODO: Create a ToolNode using our tools list
tool_node = ToolNode(tools=...)

### Exercise 4.2 – Define the Chatbot Node

Define a chatbot function that takes the `state` as input and returns a dictionary with a single key `messages`.

When invoke the LLM, we provide it with all the `messages` in `state`.

<img src="images/chatbot.png" width="800">
<img src="images/chatbot2.png" width="800">

In [ ]:
from typing import Dict, List

def chatbot(state: ChatState) -> Dict[str, List]:
    """The main chatbot node. It invokes the LLM with the current messages."""
    return {'messages': [llm_with_tools.invoke(state['messages'])]}

### Exercise 4.3 – Define the Routing Function

Create a function `route_tools` that decides whether to call tools or terminate. 

- Step 1: Extract the last message from `state`.
- Step 2: If the last message contains tool calls (`tool_calls` attribute), return `'tools'`; otherwise return `'END'`.

In [ ]:
# TODO: complete the output of route function
def route_tools(state: ChatState) -> str:
    messages = state.get('messages', [])
    ai_message = messages[-1] if messages else None
    if ai_message and getattr(ai_message, 'tool_calls', []):
        return ...
    return ...

### Exercise 4.4 – Build the Complete Graph

Assemble the agent graph by adding nodes and edges, then compile the graph into a runnable agent. Use `add_node()` for your chatbot and tool nodes, `add_conditional_edges()` for routing logic, and `add_edge()` to connect nodes.


<img src="images/graph_create_1.png" width="800">

<img src="images/graph_create_2.png" width="800">

In [ ]:
graph_builder = StateGraph(ChatState)

# TODO: Add the chatbot node to the graph
graph_builder.add_node('...', ...)

# TODO: Add the tool node to the graph
graph_builder.add_node('...', ...)

# TODO: Add conditional edges from chatbot using the routing function
graph_builder.add_conditional_edges('...', route_tools, {'...': '...', '...': ...})

# TODO: Add edge from tools back to chatbot
graph_builder.add_edge('...', '...')

# TODO: Add edge from START to chatbot
graph_builder.add_edge(..., '...')

# Compile the graph into an agent
agent = graph_builder.compile()

### Exercise 4.5 – Visualize Your Agent

Use the graph's `draw_mermaid_png()` method to visualize the structure of your agent. This step is optional but helps you understand how nodes and edges connect.

In [ ]:
from IPython.display import Image, display

try:
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception:
    print('Graph visualization not available in this environment.')

---

## Part 5 – Testing Your Agent

Now that your agent graph is built, it's time to interact with it. Create a simple chat loop that greets the user, processes input until they type 'quit', and streams responses using `agent.stream()`.

### Exercise 5.1 – Create a Basic Chat Loop

Write an interactive loop that:
1. Greets the user and explains the agent's capabilities.
2. Reads user input in a loop and exits on `'quit'`, `'exit'` or `'q'`.
3. Creates the initial `state` with the user's message and streams the agent's responses via `agent.stream()`.
4. Uses the `pretty_print()` method on messages to display nicely formatted output.


Try out this chat loop with these three prompts: 

- **Prompt 1**: ” You are an expert drug discovery researcher. Use your available tools to answer the user’s question as accurately as possible. Never fabricate or invent data. Here is the compound ID: Erlotinib. Is it orally drug-like?”


- **Prompt 2**: " You are an expert drug discovery researcher. Use your available tools to answer the user’s question as accurately as possible. Never fabricate or invent data. Which is more lipophilic: Imatinib or Dasatinib?”


- **Prompt 3**:  " You are an expert drug discovery researcher. Use your available tools to answer the user’s question as accurately as possible. Never fabricate or invent data. Check betulinic acid. Are there any structural red flags for oral bioavailability?”

- **Prompt 4**:  " You are an expert drug discovery researcher. Use your available tools to answer the user’s question as accurately as possible. Never fabricate or invent data. What are known EGFR inhibitors in the literature, and are they drug-like?”

- **Prompt 5**:  " You are an expert drug discovery researcher. Use your available tools to answer the user’s question as accurately as possible. Never fabricate or invent data. What are the common physicochemical characteristics of successful JAK inhibitors?"




In [ ]:
print('Welcome to the LangGraph ReAct demo with memory! Ask me a question.')
print('I can perform simple math, look up drug information, and search PubMed via LitSense.')
print('====================================================================')

while True:
    try:
        # Get input prompt from user
        user_input = input('User: ')
    except EOFError:
        break
    
    # If input contains one of three keywords: quit, exit, and q; exit the streaming process.
    if not user_input or user_input.lower() in {'quit', 'exit', 'q'}:
        print('Goodbye!')
        break
    
    # Initialize graph state with user's input
    state = {
        'messages': [
            {'role': 'user', 'content': user_input}
        ]
    }

    # Print the user's message first
    print(f"""================================== User's Message ==================================
{user_input}
    """)
    # Stream the answer from agent
    for event in agent.stream(state):
        for value in event.values():
            # The last message in the state is the AI response
            msg = value["messages"][-1]
            # Use the built-in pretty_print method for better formatting
            msg.pretty_print()

---

## Part 6: Adding Short‑term Memory

To demonstrate the lack of memory in the agent, please **go back to Part 5**, then perform these two prompts: 

- **Prompt 1:** Here is the compound ID: Erlotinib. Is it orally drug-like?

- **Prompt 2:** Which is more lipophilic: Imatinib or the previous drug?

You will see when invoking **prompt 2**, the agent get confuse since it doesn't know the previous drug is Erlotinib. This is because the GraphState is only maintained from the START node to the END node. After termination, all contexts from the previous run are lost. Therefore, we need a component called Agent Memory to maintain context across runs.

Memory allows your agent to remember previous parts of the conversation. LangGraph uses "checkpointers" to maintain state across interactions.

### Exercise 6.1 – Add Memory to Your Agent

Rebuild your agent with memory by creating a new `StateGraph` that uses the same `ChatState`. 

Initialize `checkpointer` with `InMemorySaver` to persist state across turns.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

class ChatState(TypedDict):
    messages: Annotated[List[Dict], add_messages]

graph_builder = StateGraph(ChatState)

llm = get_llm()
llm_with_tools = llm.bind_tools(TOOLS, parallel_tool_calls=False)

def chatbot(state: ChatState) -> Dict[str, List]:
    return {'messages': [llm_with_tools.invoke(state['messages'])]}

tool_node = ToolNode(tools=TOOLS)

graph_builder.add_node('chatbot', chatbot)
graph_builder.add_node('tools', tool_node)
graph_builder.add_conditional_edges('chatbot', route_tools, {'tools': 'tools', 'END': END})
graph_builder.add_edge('tools', 'chatbot')
graph_builder.add_edge(START, 'chatbot')

checkpointer = InMemorySaver()
agent = graph_builder.compile(checkpointer=checkpointer)

### Exercise 6.2 – Create a Memory‑Enabled Chat Loop

Write a chat loop similar to Part 5, but supply a `config` dictionary to `agent.stream()`. In the config, there are two important parameters, including `thread_id` and `recursion_limit`.

<img src="images/short_term_mem.png" width="800">

In [ ]:
# TODO: Create config for memory
config = {
    'configurable': {
        'thread_id': '...',
    },
    'recursion_limit': ...
}

In [ ]:
print('Welcome to the LangGraph ReAct demo with memory! Ask me a question.')
print('I can perform simple math, look up drug information, and search PubMed via LitSense.')
print('====================================================================')

while True:
    try:
        # Get input prompt from user
        user_input = input('User: ')
    except EOFError:
        break
    
    # If input contains one of three keywords: quit, exit, and q; exit the streaming process.
    if not user_input or user_input.lower() in {'quit', 'exit', 'q'}:
        print('Goodbye!')
        break

    # Initialize graph state with user's input
    state = {
        'messages': [
            {'role': 'user', 'content': user_input}
        ]
    }


    # Print the user's message first
    print(f"""================================== User's Message ==================================
{user_input}
    """)
    
    # TODO: Supply config for the agent stream process
    for event in agent.stream(state, config=...):
        for value in event.values():
            # The last message in the state is the AI response
            msg = value["messages"][-1]
            # Use the built-in pretty_print method for better formatting
            msg.pretty_print()

---

## Part 7 – Prebuilt Agents

LangGraph provides prebuilt agents that implement common architectures such as the ReAct pattern. These agents are quick to set up and let you focus on your tools and prompts rather than on graph wiring.

### Exercise 7.1 – Create a Prebuilt Agent

Use `create_agent` to instantiate a prebuilt ReAct agent. Provide your LLM, tools list, and a custom system prompt that instructs the agent how to behave.

#### References: [LangGraph Prebuilt Tutorial](https://docs.langchain.com/oss/python/langchain/agents)

In [ ]:
# Create config for memory
config = {
    'configurable': {
        'thread_id': 'example_conversation',
    },
    'recursion_limit': 25
}

In [ ]:
from langchain.agents import create_agent

llm = get_llm()

checkpointer = InMemorySaver()

agent = create_agent(
    model = llm,
    tools = TOOLS,
    system_prompt="You are an expert drug discovery researcher. Use your available tools to answer the user’s question as accurately as possible. Never fabricate or invent data.",
    checkpointer = checkpointer
)

In [ ]:
print('Welcome to the LangGraph ReAct demo with memory! Ask me a question.')
print('I can perform simple math, look up drug information, and search PubMed via LitSense.')
print('====================================================================')

while True:
    try:
        # Get input prompt from user
        user_input = input('User: ')
    except EOFError:
        break
    
    # If input contains one of three keywords: quit, exit, and q; exit the streaming process.
    if not user_input or user_input.lower() in {'quit', 'exit', 'q'}:
        print('Goodbye!')
        break

    # Initialize graph state with user's input
    state = {
        'messages': [
            {'role': 'user', 'content': user_input}
        ]
    }

    # Print the user's message first
    print(f"""================================== User's Message ==================================
{user_input}
    """)
    
    # Supply config for the agent stream process
    for event in agent.stream(state, config=config):
        for value in event.values():
            # The last message in the state is the AI response
            msg = value["messages"][-1]
            # Use the built-in pretty_print method for better formatting
            msg.pretty_print()

---

## Part 8 – Extension Exercises


### Exercise 8.1 – CodeAct agent

So far, our **ReAct** agent has picked from a fixed list of tools. This approach has two limitations:

1. **No tool-fixing mechanism.** The predefined tools aren't always a perfect fit for the current dataset, and a ReAct agent has no way to modify its tools or manipulate data flexibly when they fall short.
2. **Fixed tool space.** Once the tools are defined, that's all the agent has. Any task beyond the tool set is simply out of reach.

A **CodeAct** agent addresses both problems: instead of calling a predefined function, it writes code and runs it. As a result, rather than relying on a long list of specialized tools, a minimal CodeAct design needs only two:

1. `read_skill` — loads the full skill Markdown files into the LLM's context window so the agent knows what's available.
2. `python_executor` — executes Python code.

The catch is that giving an LLM free rein to generate code tends to produce unreliable output that doesn't follow any consistent standard or domain expertise. To keep the agent from reinventing the wheel each time, we give it a small library of **skills** — short Markdown recipes it can read on demand and adapt to the task at hand.


<img src="images/code_act.png" width="800">

Skills follow the standard Claude-skill layout:

```
utils/codeact_skills/skills/
├── lipinski/        ← skill-name as the dir name
│   └── SKILL.md     ← starts with YAML frontmatter: name + description
└── tanimoto/
    └── SKILL.md
```

For each skill, only `name` and `description` are injected to the agent's system prompt, so the agent know what exists. During the run time, agent dynamically decides which tools to load to the context.

In [ ]:
from pathlib import Path

# Helper function to parse SKILLs.md
SKILLS_DIR = Path("utils/codeact_skills/skills")

def _parse_frontmatter(text: str) -> dict:
    """Extract YAML frontmatter (between '---' fences) as a dict."""
    if not text.startswith("---"):
        return {}
    end = text.find("\n---", 3)
    if end == -1:
        return {}
    return dict(
        (k.strip(), v.strip())
        for line in text[3:end].strip().splitlines()
        if ":" in line
        for k, _, v in [line.partition(":")]
    )


def load_skill_index() -> list[dict]:
    """Return [{name, description, path}, ...] for every skill folder."""
    return [
        {
            "name": meta.get("name", skill_file.parent.name),
            "description": meta.get("description", ""),
            "path": skill_file,
        }
        for skill_file in sorted(SKILLS_DIR.glob("*/SKILL.md"))
        for meta in [_parse_frontmatter(skill_file.read_text())]
    ]

In [ ]:
# Parse SKILL.md and establish skill list (SKILL_CATALOG)
skill_index = load_skill_index()
SKILL_CATALOG = "\n".join(
    f"- {s['name']}: {s['description']}" for s in skill_index
)
print("Available skills:\n" + SKILL_CATALOG)

# TODO: Define tool list for CodeAct Agent:
from utils.codeact_skills.python_executor import python_executor
from utils.codeact_skills.read_files import read_skill

CODEACT_TOOLS = [.........]

In [ ]:
# System prompt for CodeAct agent
codeact_system_prompt = f"""You are a drug-discovery research assistant operating as a CodeAct agent: you reason in natural language and act by writing Python code that runs in a stateful executor. Your job is to find the smartest, fastest path to a correct, evidence-backed answer — not the most cautious one.

## Available skills
Each skill is a SKILL.md file containing domain knowledge, API conventions, and worked example code. Load a skill by calling `read_skill('<name>')`.
{SKILL_CATALOG}

Skills are your reliability backbone: when one applies, it encodes the right libraries, the right conventions, and patterns that have been tested. Prefer skill-grounded code over improvisation whenever a skill fits. But skills are a *floor*, not a *ceiling* — you are a competent Python programmer and chemist, and the catalog will not cover every task.

## Startup triage (fast, not ceremonial)
Before writing analysis code, take one short pass to plan:
1. **Identify what the task actually needs** — what data, what computation, what domain (cheminformatics, ML, statistics, file parsing, plotting, etc.).
2. **Match against the catalog.** For each relevant area, load the matching skill via `read_skill(...)` *before* writing code in that area.
3. **Decide your mode:**
   - **Skill-grounded mode** — one or more skills match. Adapt their example code; do not reinvent what a skill already specifies.
   - **Out-of-catalog mode** — no skill applies, or only part of the task is covered. Proceed with first-principles Python using standard libraries (rdkit, pandas, numpy, scipy, sklearn, matplotlib, etc. as available). When you do this, **briefly note in your final answer that this part of the work was outside the skill set** so the user knows the reliability profile.
   - **Hybrid** — load the skills that apply, write original code for the rest. State which is which when it matters.

This triage is not a wall. If midway through you realize a different skill applies, load it then. Skills are cheap.

## The execute → reason loop
Work in small, verifiable steps.

1. **Execute.** Write a short Python snippet and run it via `python_executor`. Each snippet targets ONE sub-goal (load a dataset, inspect its shape, compute one metric, generate one plot). Variables persist across executions — build state incrementally.
2. **Reason.** After each execution, examine the output before doing anything else. State in plain language: what you observed, whether it matches your expectation, what it implies, and what the next step is. If the output is an error or surprise, diagnose it before proceeding — never blindly retry or paper over it.
3. **Iterate.** Use that reasoning to decide the next execution. Repeat until you have enough evidence to answer.

### Rules for the loop
- **Inspect before you compute.** First execution on any new data (uploaded file, API response, query result) examines structure — shape, dtypes, head, schema — before any analysis runs against it.
- **One thing at a time.** Don't chain ten operations into one cell hoping they all work. Small steps localize errors and make reasoning verifiable.
- **Never answer from memory alone.** Even when you "know" the answer (a pKa, a property, a SMILES, a benchmark number), you must verify it by executing code against an authoritative source or computation in this session. Stating a value without having produced it in the loop is a failure mode, not a shortcut.
- **Never fabricate results.** If a computation fails or returns nothing useful, say so. Do not invent numbers, citations, molecules, or structures to fill the gap.

## How far to push
Push hard. The default is to keep going.

- **Don't stop on uncertainty about approach** — that's what the loop is for. Try the most plausible method; if it doesn't work, the error tells you what to do next.
- **Don't stop because a skill is missing** — fall back to first-principles code and proceed.
- **Don't ask the user clarifying questions when a reasonable interpretation exists.** Make the assumption, state it in your final answer, and proceed. Only ask when the question is genuinely ambiguous in a way that changes the answer materially.

Stop only on real blockers:
- Required data is missing or inaccessible and cannot be derived.
- The question itself is contradictory or under-specified in a way no reasonable assumption fixes.
- A hard environment failure (library unavailable, API down) that you've already tried to work around.

When you do stop short, say exactly what's blocking and what you'd need to continue.

## Stop when you have the answer
Once the evidence supports a clear answer, stop executing. Extra runs add noise and risk introducing errors.

## Final response to the user
Reply with both the reasoning chain and the critical evidence — that's what makes an answer defensible.

Include:
- **The direct answer** to what was asked, up front.
- **The reasoning** in brief prose: what approach you took, what the key intermediate findings were, and why they support the conclusion.
- **The critical evidence**: the specific numbers, structures, plots, or table excerpts that the answer rests on. Show the load-bearing values inline, not a transcript of every cell.
- **Caveats**: assumptions you made, limitations of the method, whether any part was out-of-catalog, anything the user should know before acting on the result.

Do not narrate the loop or list every skill you loaded. The user wants a conclusion they can trust and verify, not a diary."""

In [ ]:
from langchain.agents import create_agent

# TODO: Define the CodeAct agent 
codeact_agent = create_agent(
    model=get_llm(),
    tools=...,
    system_prompt=....,
)

In [ ]:
# Run the CodeAct agent and watch each step
question = "request-the-agent-to-do-something-here"


print(f"USER: {question}\n")
for event in codeact_agent.stream({'messages': question}):
    for value in event.values():
            msg = value["messages"][-1]
            msg.pretty_print()

### Exercise 8.5 – Multi-agent system with a supervisor

One agent with many tools can get confused. A cleaner pattern: a **supervisor** that delegates each question to a specialist **sub-agent**. We build three sub-agents and one supervisor by hand so you can see exactly how the routing works.

**Sub-agents** (each is just `create_react_agent` with its own prompt and tool set):

- **MedChem expert** — `resolve_smiles`, `get_properties`. Answers "what does this molecule look like / does it pass drug-likeness rules".
- **Physiology expert** — `lit_search`. Answers mechanism / disease-biology questions from PubMed.
- **Data Analysis expert** — `python_executor`. Runs small computations on numbers the user provides.

**Supervisor** — a plain ReAct agent whose tools are *the sub-agents themselves*, wrapped as functions. The supervisor reads the user's question, picks one specialist, and forwards the query. This "agent-as-tool" trick keeps the design tiny and easy to read.

<img src="images/supervisor.png" width="800">

**IMPROTANT:** This is a minimal toy example meant to illustrate how to build a multi-agent system. Making it production-ready would require substantial work — building proper tools, tuning system prompts, and doing careful context engineering both within and across agents.


In [ ]:
# Build the three specialist sub-agents
from langchain.agents import create_agent

llm_specialist = get_llm()

medchem_agent = create_agent(
    model=llm_specialist,
    tools=[resolve_smiles, get_properties],
    system_prompt=(
        "You are a medicinal chemist. Resolve identifiers to SMILES with `resolve_smiles` "
        "and compute properties with `get_properties`. Reply concisely with the numbers."
    ),
)

physiology_agent = create_agent(
    model=llm_specialist,
    tools=[lit_search],
    system_prompt=(
        "You are a physiology / pharmacology expert. Use `lit_search` to find evidence in "
        "PubMed, then summarise the mechanism in 2-3 sentences and cite PMIDs."
    ),
)

data_agent = create_agent(
    model=llm_specialist,
    tools=[python_executor],
    system_prompt=(
        "You are a data analyst. Use `python_executor` to run small computations "
        "(stats, plots, simple maths). Reply with the final number or summary only."
    ),
)

In [ ]:
# Wrap each sub-agent as a tool the supervisor can call
from langchain.tools import tool

def _ask(agent, query: str) -> str:
    result = agent.invoke({'messages': query})
    return result['messages'][-1].content

@tool
def ask_medchem(query: str) -> str:
    """Delegate to the medicinal chemistry expert (SMILES, molecular properties, drug-likeness)."""
    print(f"  -> routing to MedChem: {query}")
    return _ask(medchem_agent, query)

@tool
def ask_physiology(query: str) -> str:
    """Delegate to the physiology / pharmacology expert (mechanism of action, disease biology, PubMed)."""
    print(f"  -> routing to Physiology: {query}")
    return _ask(physiology_agent, query)

@tool
def ask_data_analyst(query: str) -> str:
    """Delegate to the data analyst (numerical computation, simple stats)."""
    print(f"  -> routing to Data Analyst: {query}")
    return _ask(data_agent, query)


In [ ]:
# 8.5 – Build the supervisor
supervisor_prompt = (
    "You are a research supervisor coordinating three specialists:\n"
    "  - ask_medchem      : SMILES / molecular properties / drug-likeness\n"
    "  - ask_physiology   : mechanism of action, disease biology, PubMed evidence\n"
    "  - ask_data_analyst : numerical computation\n"
    "For each user question, decide which specialist (or combination) is needed, "
    "call them, then combine their answers into one short final reply for the user. "
    "Do not answer scientific questions yourself — always delegate."
)

supervisor = create_agent(
    model=get_llm(),
    tools=[ask_medchem, ask_physiology, ask_data_analyst],
    system_prompt=supervisor_prompt,
)


In [ ]:
# Try the supervisor on a question that needs more than one specialist
question = (
    "ask-the-agent-to-do-something-here"
)

print(f"USER: {question}\n")
for step, event in enumerate(supervisor.stream({'messages': question}, stream_mode='values'), start=1):
    last = event['messages'][-1]
    if getattr(last, 'tool_calls', None):
        for tc in last.tool_calls:
            print(f"[step {step}] supervisor calling {tc['name']}")
    else:
        preview = (last.content or '').strip().replace('\n', ' ')
        if len(preview) > 250:  # Modify here if you want to see full messages or longer messages
            preview = preview[:250] + '...'
        print(f"[step {step}] {type(last).__name__}: {preview}")


---

## Key Takeaways

- **Nodes** are functions that operate on state.
- **Edges** define the flow between nodes.
- **State** carries data through the agent. Use reducer functions such as `add_messages` to control how the state is updated.
- **Tools** extend agent capabilities and are annotated with `@tool`.
- **Memory** allows agents to maintain context across invocations.
- **Prebuilt agents** provide quick solutions for common patterns but offer less control than a custom graph.

---
## Resources

- **LangChain Documentation** – https://docs.langchain.com/oss/python/langchain/overview
- **LangGraph Documentation** – https://docs.langchain.com/oss/python/langgraph/overview
- **OpenAI API** – https://platform.openai.com/docs/
- **ReAct agent** - https://arxiv.org/abs/2210.03629
- **CodeAct agent** - https://arxiv.org/abs/2402.01030

These references can help you explore LangGraph and related libraries beyond the scope of this workshop.

---

**🎉 Congratulations! You've built your first AI agent with LangGraph!**

Feel free to modify and extend your agent. You can experiment with new tools, different LLMs, and additional nodes to create even more capable and personalised agents.